# Synthetic data generation -- Kaggle runner

Generates new Task 1 + Task 2 training data via controlled bug injection (see
[METHOD.md](https://github.com/ptran1203/msc/blob/main/llm/METHOD.md) section 4 for the
design: the LLM only writes buggy code from a precise instruction, labels come from an
objective oracle -- never from the LLM self-grading its own output).

Attach the private dataset `ptran1203/llm-autograding-course-data` under Notebook -> Add Input before running.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
%cd /kaggle/working
!rm -rf msc
!git clone --depth 1 https://github.com/ptran1203/msc.git
%cd /kaggle/working/msc/llm
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -y -qq g++ > /dev/null
!g++ --version


In [ ]:
import shutil, pathlib
dataset_root = pathlib.Path("/kaggle/input/datasets/ptran1203/llm-autograding-course-data")
repo_llm = pathlib.Path("/kaggle/working/msc/llm")
shutil.copytree(dataset_root / "data", repo_llm / "data")
print("data staged OK")


## Smoke test -- oracle self-check, then 1 bug type x 1 variant x 5 problems

Per the no-heavy-compute-without-checking-first rule: verify the oracle (compile/run/compare,
no model needed) and the full LLM pipeline both work before the real run.

In [ ]:
%cd /kaggle/working/msc/llm/src
!python synth_oracle.py


In [ ]:
!python synth_generate.py --smoke-test


## Full run -- 5 problems x 10 bug types x 20 variants (up to 1000 candidates before QC discards)

Two new problems added (binary_search, palindrome_check) beyond the original 3
(armstrong, remove_duplicates, earliest_login) -- the only other real single-problem
exam (Class03-Midterm-1) turned out to be a duplicate Armstrong instance, and the
remaining real exams are a different struct + file-I/O + linked-list family with no
stdin/stdout main(), too big a lift to retrofit here -- so these two synthesize their
own exam metadata instead of reusing a real exam's.

Variants >=1 use seeded sampling (not greedy), so they actually diversify instead of
repeating near-identical edits -- locally this roughly doubled yield at variants=6 (57->109
kept) and even got `compile_error` producing usable samples at all (0/9 under pure greedy).
Pushing variants higher for maximum yield; some bug types (e.g. `style_error`) will likely
saturate toward 100% long before 20, others (e.g. `edge_case_error`) have more headroom.
Edit `--variants` to generate more/fewer per bug type.

In [ ]:
!python -u synth_generate.py --variants 20


## Collect output + clean up

Copies just the generated data out of the repo clone into `/kaggle/working/synth_generated/`,
then deletes the repo -- so the notebook's saved output is only the synthetic data, not the
whole source tree + per-candidate submission files.

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/synth_generated", ignore_errors=True)
shutil.copytree("/kaggle/working/msc/llm/synth/generated", "/kaggle/working/synth_generated")
%cd /kaggle/working
shutil.rmtree("/kaggle/working/msc", ignore_errors=True)
print("kept files:", len(list(__import__("pathlib").Path("/kaggle/working/synth_generated").rglob("*"))))


## Output

`/kaggle/working/synth_generated/` (schema-compatible with `sample_dataset`, loadable via
`data.load_dataset()`).

**Before using this data elsewhere**: it embeds real exam statements (course content), so per
the course rules it should go back into the private Kaggle dataset
(`ptran1203/llm-autograding-course-data`), never into the public code repo or anywhere public -- re-upload it there,
don't just leave it only as a kernel output.